In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

vocab_size = 20
seq_len = 5
embed_dim = 8
num_heads = 2
num_classes = 2

x = torch.tensor([
    [1,2,3,4,5],
    [6,7,8,9,10],
    [11,12,13,14,15]
])

embeddings = nn.Embedding(vocab_size,embed_dim)
x_embed = embeddings(x)


print("token embedding shape -> ",x_embed.shape)

token embedding shape ->  torch.Size([3, 5, 8])


In [15]:
x_embed[0]

tensor([[-2.6527,  1.8773, -0.3676, -1.7496, -0.4410, -1.2903, -0.5940, -0.3315],
        [ 0.0402,  1.1933,  0.7914,  0.4919, -2.3050,  1.8350,  1.5763,  3.7072],
        [ 1.3149,  1.4170, -0.1952,  1.1272, -1.1636, -1.5492, -1.3876,  0.3373],
        [-1.5440, -1.2962,  0.7209,  1.5324,  1.3797, -0.2162, -0.2721, -0.4491],
        [-1.2958,  0.2357, -2.9259, -2.1306, -0.8693,  2.7547, -0.9221, -0.5214]],
       grad_fn=<SelectBackward0>)

In [3]:
positional_embeddings = nn.Embedding(seq_len,embed_dim)
positions = torch.arange(seq_len)
pos_embed = positional_embeddings(positions)

x_embed = x_embed + pos_embed

x_embed[0]

tensor([[-2.6527,  1.8773, -0.3676, -1.7496, -0.4410, -1.2903, -0.5940, -0.3315],
        [ 0.0402,  1.1933,  0.7914,  0.4919, -2.3050,  1.8350,  1.5763,  3.7072],
        [ 1.3149,  1.4170, -0.1952,  1.1272, -1.1636, -1.5492, -1.3876,  0.3373],
        [-1.5440, -1.2962,  0.7209,  1.5324,  1.3797, -0.2162, -0.2721, -0.4491],
        [-1.2958,  0.2357, -2.9259, -2.1306, -0.8693,  2.7547, -0.9221, -0.5214]],
       grad_fn=<SelectBackward0>)

In [17]:
pos_embed[0]

tensor([-1.9006,  0.2286,  0.0249, -0.3460,  0.2868, -0.7308,  0.1748, -1.0939],
       grad_fn=<SelectBackward0>)

In [4]:
# Transformers

encoder_layer = nn.TransformerEncoderLayer(
    d_model = embed_dim,
    nhead = num_heads,
    dim_feedforward = 32,
    dropout = 0.1,
    batch_first = True
)

transformer_encoder = nn.TransformerEncoder(
    encoder_layer = encoder_layer,
    num_layers = 2
)

x_out = transformer_encoder(x_embed)

print("Transformer output shape -> ",x_out.shape)

Transformer output shape ->  torch.Size([3, 5, 8])


In [5]:
# Mean Pooling

x_pooled = x_out.mean(dim=1)

print("Pooled Output : ",x_pooled.shape)

Pooled Output :  torch.Size([3, 8])


In [6]:
# Classification head

classification_head = nn.Linear(embed_dim,num_classes)

logits = classification_head(x_pooled)

print("Logits shape -> ",logits.shape)

prediction = logits.argmax(dim=1)

print("Prediction : ",prediction)


Logits shape ->  torch.Size([3, 2])
Prediction :  tensor([0, 1, 1])


In [7]:
# full trainable model

class SimpleTransformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.token_embedding = nn.Embedding(20,8)
    self.pos_embedding = nn.Embedding(5,8)

    layer = nn.TransformerEncoderLayer(
        d_model = 8,
        nhead = 2,
        dim_feedforward = 32,
        dropout = 0.1,
        batch_first = True
    )

    self.encoder = nn.TransformerEncoder(
        encoder_layer = layer,
        num_layers = 2
    )

    self.classification_head = nn.Linear(8,2)

  def forward(self,x):
    x_embed = self.token_embedding(x)

    # Fix: Create positions on the same device as input x
    positions = torch.arange(x.shape[1], device=x.device)
    pos_embed = self.pos_embedding(positions)

    x_embed = x_embed + pos_embed

    x_out = self.encoder(x_embed)

    x_pooled = x_out.mean(dim=1)

    logits = self.classification_head(x_pooled)
    return logits

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleTransformer()
model = model.to(device)
print(model)
print(model.cuda)


SimpleTransformer(
  (token_embedding): Embedding(20, 8)
  (pos_embedding): Embedding(5, 8)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=8, out_features=8, bias=True)
        )
        (linear1): Linear(in_features=8, out_features=32, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=32, out_features=8, bias=True)
        (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classification_head): Linear(in_features=8, out_features=2, bias=True)
)
<bound method Module.cuda of SimpleTransformer(
  (token_embedding): Embedding(20, 8)
  (pos_embedding): Embedding(5, 8

In [9]:
optimizer = torch.optim.Adam(model.parameters(),lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [10]:
X = torch.randint(0,20,(1000,5))
y = (X == 7).any(dim=1).long()
X = X.to(device)
y = y.to(device)
print(X.device)
print(y.device)

cpu
cpu


In [11]:
for epoch in range(10):
  logits = model(X)

  loss = criterion(logits,y)

  loss.backward()
  optimizer.step()
  optimizer.zero_grad()

  print(f"Epoch : {epoch+1} | Loss : {loss.item():.4f}")

  with torch.no_grad():
    logits = model(X)
    predictions = logits.argmax(dim=1)
    accuracy = (predictions == y).float().mean()
  print(f"Epoch : {epoch+1} | Accuracy : {accuracy.item():.4f}")

Epoch : 1 | Loss : 0.6242
Epoch : 1 | Accuracy : 0.7190
Epoch : 2 | Loss : 0.6245
Epoch : 2 | Accuracy : 0.7120
Epoch : 3 | Loss : 0.6227
Epoch : 3 | Accuracy : 0.7110
Epoch : 4 | Loss : 0.6214
Epoch : 4 | Accuracy : 0.7200
Epoch : 5 | Loss : 0.6223
Epoch : 5 | Accuracy : 0.7120
Epoch : 6 | Loss : 0.6201
Epoch : 6 | Accuracy : 0.7240
Epoch : 7 | Loss : 0.6196
Epoch : 7 | Accuracy : 0.7290
Epoch : 8 | Loss : 0.6196
Epoch : 8 | Accuracy : 0.7310
Epoch : 9 | Loss : 0.6186
Epoch : 9 | Accuracy : 0.7230
Epoch : 10 | Loss : 0.6178
Epoch : 10 | Accuracy : 0.7270


In [18]:
# Vision Transformer

import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Patch Embeddings
# 224x224x3 -> sequence of path vectors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PatchEmbedding(nn.Module):
  def __init__(self,img_size=224,patch_size=16,in_channels=3,embed_dim=256):
    super().__init__()

    self.img_size = img_size
    self.patch_size = patch_size
    # 224x224x3 -> (224/16) x (224/16) -> 14x14 -> 196
    self.num_patchs = (img_size // patch_size) ** 2
    self.projection = nn.Conv2d(
        in_channels = in_channels,
        out_channels = embed_dim,
        kernel_size = patch_size,
        stride = patch_size
    )

  def forward(self,x):
    # x : [B,3,224,224]
    x = self.projection(x) # [B,embed_dim,14,14]
    x = x.flatten(2) # [B,embed_dim,196]
    x = x.transpose(1,2) # [B,196,embed_dim]
    return x

In [19]:
# Transformer Encoder Block

class TransformerEncoderBlock(nn.Module):
  def __init__(self,embed_dim=256,num_heads=4,mlp_ratio=4,dropout=0.1):

    super().__init__()

    self.norm1 = nn.LayerNorm(embed_dim)
    self.attn = nn.MultiheadAttention(embed_dim,num_heads,dropout,batch_first=True)
    self.norm2 = nn.LayerNorm(embed_dim)
    self.mlp = nn.Sequential(
        nn.Linear(embed_dim,mlp_ratio*embed_dim),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(mlp_ratio*embed_dim,embed_dim),
        nn.Dropout(dropout)
    )

  def forward(self,x):
    # x: [B,196(num_patches),256(embed_dim)]

    x_norm = self.norm1(x)
    attn_out,_ = self.attn(x_norm,x_norm,x_norm) # self attention
    x = x + attn_out # skip connection

    x_norm = self.norm2(x)
    mlp_out = self.mlp(x_norm)
    x = x+mlp_out # skip connection

    return x


In [20]:
# Vision Transformer
# CLS Token , Pos Embedding, Encoder Blocks, Classification head

class VisionTransformer(nn.Module):
  def __init__(
      self,
      img_size = 224,
      patch_size = 16,
      in_channels = 3,
      embed_dim = 256,
      num_heads = 4,
      mlp_ratio = 4,
      depth = 5,
      dropout = 0.1,
      num_classes = 4
  ):

    super().__init__()
    self.patch_embedding = PatchEmbedding(
        img_size = img_size,
        patch_size = patch_size,
        in_channels = in_channels,
        embed_dim = embed_dim
    )

    num_patches = self.patch_embedding.num_patchs

    self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))

    self.pos_embed = nn.Parameter(torch.randn(1,num_patches+1,embed_dim))

    # transformer encoder block

    self.blocks = nn.ModuleList([
        TransformerEncoderBlock(
            embed_dim = embed_dim,
            num_heads = num_heads,
            mlp_ratio = mlp_ratio,
            dropout = dropout
        ) for _ in range(depth)
    ])

    self.norm = nn.LayerNorm(embed_dim)

    self.head = nn.Linear(embed_dim,num_classes)


  def forward(self,x):
    # x : [B,3,224,224]

    B = x.shape[0]

    # get the patch embedding

    x = self.patch_embedding(x) # [B,196,256]

    # add the cls token

    cls_token = self.cls_token.repeat(B,1,1) # [B,1,256]
    x = torch.cat([cls_token,x],dim=1) # [B,197,256]

    # add the positional embedding

    x = x + self.pos_embed[:,: x.size(1),:]

    # pass through transformer encoder

    for blk in self.blocks:
      x = blk(x)

    x = self.norm(x)

    # take the cls token

    cls_repr = x[:,0] # [B,D]

    # pass through the classification head

    logits = self.head(cls_repr)

    return logits


dummy_img = torch.randn(8,3,224,224).to(device)

model = VisionTransformer()

model = model.to(device)

logits = model(dummy_img)

print(logits.shape) # [8,4]


torch.Size([8, 4])


In [21]:
# hyper-parameters
import torch.optim as optim

num_epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = 0.001)

In [22]:
# Transfer Learning


import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

100%|███████████████████████████████████████| 157M/157M [00:10<00:00, 15.6MB/s]

Extracting files...


Path to dataset files: /Users/mukundanramesh/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2


In [ ]:
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

img_size = 224

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = True,
    download = True,
    transform = train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = False,
    download = True,
    transform = test_transform
)

train_loaders = torch.utils.data.DataLoader(
    train_dataset,
    batch_size = 128,
    shuffle = True
)

test_loaders = torch.utils.data.DataLoader(
    test_dataset,
    batch_size =128,
    shuffle = False
)


  1%|▍                                   | 2.16M/170M [00:53<1:03:29, 44.2kB/s]

In [ ]:
def train_loop(model, train_loader, criterion, optimizer):
  model.train()

  running_loss = 0
  correct = 0
  total = 0

  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    # Prediction
    logits = model(images)

    # Compute the loss
    loss = criterion(logits,labels)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # Weight update
    optimizer.step()

    running_loss += loss.item()

    predictions = torch.argmax(logits,dim = 1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

  epoch_loss = running_loss / len(train_loader)
  epoch_acc = correct / total

  print(f"Train Loss : {epoch_loss:.4f} | Train Acc : {epoch_acc:.4f}")



for epoch in range(num_epochs):
  print(f"Epoch : {epoch+1}")
  train_loop(model,train_loaders,criterion,optimizer)


In [ ]:
# Test Acc

def test_loop(model,test_loader):
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for images,labels in test_loader:

      images = images.to(device)
      labels = labels.to(device)

      logits = model(images)

      predictions = torch.argmax(logits,dim = 1)

      correct += (predictions == labels).sum().item()
      total += labels.size(0)

  epoch_acc = correct / total

  print(f"Test Acc : {epoch_acc:.4f}")

test_loop(model,test_loaders)